# CoHiM-FL — Combine Five Independent Seeds

Run this notebook **only after all five seed notebooks have completed**. It reads each isolated seed result folder, performs strict completeness/provenance checks, then computes publication-ready **Mean ± sample Standard Deviation (ddof=1)**. It also combines the component-wise ablation study and creates final figures, Excel tables, and a ZIP archive.

Required seeds: **7, 21, 42, 84, 126**. No best-seed selection is performed.

## Archived results and reproducibility notice

This repository includes **original result files without numeric modifications**. The archived run used the method identifier `Proposed_CBA_NHFL_MI_Q1`, whereas these notebooks now use `Proposed_CoHiM_FL`. The aggregation notebook transparently maps only this identifier when it reads archived CSVs. This is a **label-compatibility change**, not evidence that a re-run of the renamed code has been completed. Before a reproducibility claim, re-run from the archived environment, verify GPU/software dependencies, and check generated results.

- Five-seed archived results: `results/CoHiM-FL_Split_Seeds/`
- Weight sensitivity: `results/CoHiM-FL_Weight_Sensitivity_5Seeds/`
- Audit limitations and guide: `README.md`, `reports/RESULTS_AUDIT.md`


In [ ]:
# =========================================================
# 1) Setup and strict reproducibility contract
# =========================================================
!pip -q install pandas numpy matplotlib openpyxl

import os, glob, zipfile
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REQUIRED_SEEDS = [7, 21, 42, 84, 126]
CLIENT_SETTINGS = [5, 10, 20, 40]
METHODS = ["FedAvg", "FedProx", "Fedsw", "CAR-MFL", "Proposed_CoHiM_FL"]
ABLATION_CLIENT_SETTINGS = [20]
ABLATION_VARIANTS = [
    "Full_Proposed",
    "w_o_Client_Selection",
    "w_o_Hierarchical",
    "w_o_Adaptive_TopK",
    "w_o_INT8",
    "w_o_Error_Feedback",
    "w_o_Missing_Reconstruction",
]

# Historical outputs predate the project display-name change. Map LABELS ONLY.
# Original source CSV/JSON files are kept unmodified on disk.
LEGACY_METHOD_ALIASES = {"Proposed_CBA_NHFL_MI_Q1": "Proposed_CoHiM_FL"}
def canonicalize_legacy_labels(df):
    if "Method" in df.columns:
        df = df.copy()
        df["Method"] = df["Method"].replace(LEGACY_METHOD_ALIASES)
    return df

SPLIT_RUN_ROOT = "/content/drive/MyDrive/CoHiM-FL_Split_Seeds"
COMBINED_DIR = os.path.join(SPLIT_RUN_ROOT, "combined_5_seeds")
FIGURE_DIR = os.path.join(COMBINED_DIR, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

PUBLICATION_METRICS = [
    "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
    "Label_F1_micro", "Label_F1_weighted",
    "Jaccard_micro", "Hamming_Loss",
    "Total_Comm_Cost_MB", "Cost_Reduction_%",
]

print("Required seeds:", REQUIRED_SEEDS)
print("Input root:", SPLIT_RUN_ROOT)
print("Combined output:", COMBINED_DIR)


In [ ]:
# =========================================================
# 1B) Strict data-integrity verification across all five seeds
# =========================================================
# The aggregation notebook refuses to compute publication Mean ± SD unless:
#   - every seed produced a PASS leakage audit,
#   - all UID/image overlaps are zero,
#   - every seed used exactly the SAME fixed Train/Validation/Test partition,
#   - patient-level status is consistent across runs.
# =========================================================

import json

split_audit_rows = []
split_fingerprints = set()
patient_statuses = set()

for seed in REQUIRED_SEEDS:
    audit_path = os.path.join(
        SPLIT_RUN_ROOT,
        f"seed_{seed}",
        "data_integrity",
        f"data_split_leakage_audit_seed_{seed}.json",
    )

    if not os.path.isfile(audit_path):
        raise FileNotFoundError(
            f"Missing required data-integrity audit for seed {seed}: {audit_path}\n"
            "Run the complete seed notebook from the beginning before aggregation."
        )

    with open(audit_path, "r", encoding="utf-8") as f:
        audit = json.load(f)

    if audit.get("overall_status") != "PASS":
        raise RuntimeError(
            f"Seed {seed} leakage audit did not PASS: {audit.get('overall_status')}"
        )

    if int(audit.get("run_seed")) != int(seed):
        raise RuntimeError(
            f"Seed provenance mismatch in leakage audit: expected {seed}, "
            f"found {audit.get('run_seed')}"
        )

    for category in ["uid_overlap", "image_file_overlap"]:
        values = audit.get(category, {})
        if any(int(v) != 0 for v in values.values()):
            raise RuntimeError(
                f"Seed {seed} has non-zero {category}: {values}"
            )

    fp = audit.get("split_fingerprint_sha256")
    if not fp:
        raise RuntimeError(
            f"Seed {seed} leakage audit is missing split fingerprint."
        )

    split_fingerprints.add(fp)
    patient_statuses.add(audit.get("patient_audit_status"))

    split_audit_rows.append({
        "Seed": seed,
        "Audit_Status": audit.get("overall_status"),
        "Data_Split_Seed": audit.get("data_split_seed"),
        "Train_Rows": audit.get("train_rows"),
        "Validation_Rows": audit.get("validation_rows"),
        "Test_Rows": audit.get("test_rows"),
        "UID_Train_Val_Overlap": audit["uid_overlap"]["train_validation"],
        "UID_Train_Test_Overlap": audit["uid_overlap"]["train_test"],
        "UID_Val_Test_Overlap": audit["uid_overlap"]["validation_test"],
        "Image_Train_Val_Overlap": audit["image_file_overlap"]["train_validation"],
        "Image_Train_Test_Overlap": audit["image_file_overlap"]["train_test"],
        "Image_Val_Test_Overlap": audit["image_file_overlap"]["validation_test"],
        "Patient_Audit_Status": audit.get("patient_audit_status"),
        "Patient_ID_Source": audit.get("patient_identifier_source"),
        "Patient_ID_Column": audit.get("patient_identifier_column"),
        "Split_Fingerprint_SHA256": fp,
        "Audit_JSON": audit_path,
    })

if len(split_fingerprints) != 1:
    raise RuntimeError(
        "The five runs did NOT use the same fixed Train/Validation/Test split. "
        f"Found {len(split_fingerprints)} distinct split fingerprints."
    )

if len(patient_statuses) != 1:
    raise RuntimeError(
        "Patient-level audit status is inconsistent across the five runs: "
        f"{sorted(patient_statuses)}"
    )

split_audit_manifest = pd.DataFrame(split_audit_rows)

print("\nDATA-INTEGRITY AUDIT ACROSS ALL FIVE SEEDS: PASS")
print("Identical fixed split fingerprint:", next(iter(split_fingerprints)))
print("Patient-level audit status:", next(iter(patient_statuses)))
display(split_audit_manifest)


In [ ]:
# =========================================================
# 2) Load and validate all five MAIN experiment result files
# =========================================================
main_frames = []
main_file_manifest = []

for seed in REQUIRED_SEEDS:
    path = os.path.join(
        SPLIT_RUN_ROOT, f"seed_{seed}", "main_results", f"final_test_seed_{seed}.csv"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing required seed result: {path}\n"
            "Do not compute Mean ± SD until all five seed notebooks are complete."
        )
    df = pd.read_csv(path)
    df = canonicalize_legacy_labels(df)

    # Provenance and completeness checks.
    if "Seed" not in df.columns:
        raise RuntimeError(f"Seed column missing in {path}")
    found = set(df["Seed"].astype(int).unique())
    if found != {seed}:
        raise RuntimeError(f"Seed provenance mismatch in {path}: found {found}, expected {{{seed}}}")

    expected_pairs = {(m, c) for c in CLIENT_SETTINGS for m in METHODS}
    actual_pairs = set(zip(df["Method"].astype(str), df["Clients"].astype(int)))
    if actual_pairs != expected_pairs:
        missing = sorted(expected_pairs - actual_pairs)
        extra = sorted(actual_pairs - expected_pairs)
        raise RuntimeError(
            f"Incomplete/invalid main results for seed {seed}. Missing={missing}; Extra={extra}"
        )

    if len(df) != len(expected_pairs):
        raise RuntimeError(f"Seed {seed}: expected {len(expected_pairs)} rows, found {len(df)}")

    # Recompute cost reduction from raw communication cost to verify the saved value.
    fedavg = (
        df[df["Method"] == "FedAvg"][["Clients", "Total_Comm_Cost_MB"]]
        .rename(columns={"Total_Comm_Cost_MB": "__FedAvg_Comm"})
    )
    check = df.merge(fedavg, on="Clients", how="left", validate="many_to_one")
    recomputed = (1.0 - check["Total_Comm_Cost_MB"] / check["__FedAvg_Comm"]) * 100.0
    if "Cost_Reduction_%" in check.columns:
        if not np.allclose(check["Cost_Reduction_%"], recomputed, rtol=1e-8, atol=1e-8, equal_nan=True):
            raise RuntimeError(f"Communication reduction audit failed for seed {seed}")
    else:
        df["Cost_Reduction_%"] = recomputed.to_numpy()

    main_frames.append(df)
    main_file_manifest.append({"Seed": seed, "Path": path, "Rows": len(df)})

final_test_all_seeds = pd.concat(main_frames, ignore_index=True)
expected_total = len(REQUIRED_SEEDS) * len(CLIENT_SETTINGS) * len(METHODS)
if len(final_test_all_seeds) != expected_total:
    raise RuntimeError(f"Expected {expected_total} combined rows, found {len(final_test_all_seeds)}")

print("MAIN RESULT AUDIT: PASSED")
display(pd.DataFrame(main_file_manifest))


In [ ]:
# =========================================================
# 3) Publication summary — Mean ± sample SD across five seeds
# =========================================================
missing_metrics = [m for m in PUBLICATION_METRICS if m not in final_test_all_seeds.columns]
if missing_metrics:
    raise RuntimeError("Missing publication metrics: " + ", ".join(missing_metrics))

agg = (
    final_test_all_seeds
    .groupby(["Method", "Clients"])[PUBLICATION_METRICS]
    .agg(["mean", "std"])
)
agg.columns = [
    f"{metric}_{'Mean' if stat == 'mean' else 'SD'}"
    for metric, stat in agg.columns
]
main_summary_numeric = agg.reset_index()
main_summary_numeric.insert(2, "N_Seeds", len(REQUIRED_SEEDS))
main_summary_numeric.insert(3, "Seeds", ",".join(map(str, REQUIRED_SEEDS)))

# Strict SD audit: all groups must contain all five independent seeds.
counts = final_test_all_seeds.groupby(["Method", "Clients"])["Seed"].nunique()
if not (counts == len(REQUIRED_SEEDS)).all():
    raise RuntimeError("At least one Method × Clients group does not contain all five seeds.")

def mean_sd_string(mean_value, sd_value, decimals=4):
    if pd.isna(mean_value) or pd.isna(sd_value):
        return "NA"
    return f"{float(mean_value):.{decimals}f} ± {float(sd_value):.{decimals}f}"

main_publication_table = main_summary_numeric[["Method", "Clients", "N_Seeds", "Seeds"]].copy()
for metric in PUBLICATION_METRICS:
    main_publication_table[metric] = [
        mean_sd_string(m, sd)
        for m, sd in zip(
            main_summary_numeric[f"{metric}_Mean"],
            main_summary_numeric[f"{metric}_SD"],
        )
    ]

print("\nPRIMARY PUBLICATION TABLE — MEAN ± SAMPLE SD (n=5)")
display(main_publication_table.sort_values(["Clients", "Method"]).reset_index(drop=True))


In [ ]:
# =========================================================
# 4) Load, validate, and combine the five ABLATION result files
# =========================================================
ablation_frames = []
ablation_manifest = []

for seed in REQUIRED_SEEDS:
    path = os.path.join(
        SPLIT_RUN_ROOT, f"seed_{seed}", "ablation_results", f"ablation_final_seed_{seed}.csv"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing ablation result for seed {seed}: {path}\n"
            "Do not report ablation Mean ± SD until all five seed ablations are complete."
        )
    df = pd.read_csv(path)
    df = canonicalize_legacy_labels(df)
    found = set(df["Seed"].astype(int).unique())
    if found != {seed}:
        raise RuntimeError(f"Ablation seed provenance mismatch in {path}: {found}")

    expected_pairs = {(c, v) for c in ABLATION_CLIENT_SETTINGS for v in ABLATION_VARIANTS}
    actual_pairs = set(zip(df["Clients"].astype(int), df["Ablation_Variant"].astype(str)))
    if actual_pairs != expected_pairs:
        raise RuntimeError(
            f"Ablation completeness failure for seed {seed}. "
            f"Missing={sorted(expected_pairs-actual_pairs)}; Extra={sorted(actual_pairs-expected_pairs)}"
        )
    ablation_frames.append(df)
    ablation_manifest.append({"Seed": seed, "Path": path, "Rows": len(df)})

ablation_all_seeds = pd.concat(ablation_frames, ignore_index=True)

ABLATION_METRICS = [
    "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
    "Label_F1_micro", "Label_F1_weighted",
    "Jaccard_micro", "Hamming_Loss", "Total_Comm_Cost_MB",
    "BLEU1_Contribution_vs_Variant",
    "METEOR_Contribution_vs_Variant",
    "Comm_Change_MB_vs_Full", "Comm_Change_%_vs_Full",
]

missing_ab = [m for m in ABLATION_METRICS if m not in ablation_all_seeds.columns]
if missing_ab:
    raise RuntimeError("Missing ablation metrics: " + ", ".join(missing_ab))

ab_agg = (
    ablation_all_seeds
    .groupby(["Clients", "Ablation_Variant"])[ABLATION_METRICS]
    .agg(["mean", "std"])
)
ab_agg.columns = [
    f"{metric}_{'Mean' if stat == 'mean' else 'SD'}"
    for metric, stat in ab_agg.columns
]
ablation_summary_numeric = ab_agg.reset_index()
ablation_summary_numeric.insert(2, "N_Seeds", len(REQUIRED_SEEDS))
ablation_summary_numeric.insert(3, "Seeds", ",".join(map(str, REQUIRED_SEEDS)))

# Component switch table comes directly from the raw records.
switch_cols = [
    "Ablation_Variant", "Adaptive_Client_Selection", "Hierarchical_Aggregation",
    "Adaptive_TopK", "INT8_Quantization", "Error_Feedback", "Missing_Reconstruction",
]
switch_table = ablation_all_seeds[switch_cols].drop_duplicates()
if switch_table["Ablation_Variant"].duplicated().any():
    raise RuntimeError("Ablation component-switch definitions differ across seeds.")
ablation_summary_numeric = ablation_summary_numeric.merge(
    switch_table, on="Ablation_Variant", how="left", validate="many_to_one"
)

ablation_publication_table = ablation_summary_numeric[[
    "Clients", "Ablation_Variant", "N_Seeds", "Seeds",
    "Adaptive_Client_Selection", "Hierarchical_Aggregation", "Adaptive_TopK",
    "INT8_Quantization", "Error_Feedback", "Missing_Reconstruction",
]].copy()
for metric in [
    "BLEU1", "METEOR", "Label_F1_micro", "Total_Comm_Cost_MB",
    "BLEU1_Contribution_vs_Variant", "METEOR_Contribution_vs_Variant",
    "Comm_Change_%_vs_Full",
]:
    ablation_publication_table[metric] = [
        mean_sd_string(m, sd)
        for m, sd in zip(
            ablation_summary_numeric[f"{metric}_Mean"],
            ablation_summary_numeric[f"{metric}_SD"],
        )
    ]

print("ABLATION AUDIT: PASSED")
display(pd.DataFrame(ablation_manifest))
print("\nABLATION PUBLICATION TABLE — MEAN ± SAMPLE SD (n=5)")
display(ablation_publication_table.sort_values(["Clients", "Ablation_Variant"]).reset_index(drop=True))


In [ ]:
# =========================================================
# 5) Publication-quality Mean ± SD figures
# =========================================================
DISPLAY_NAMES = {
    "FedAvg": "FedAvg",
    "FedProx": "FedProx",
    "Fedsw": "Fedsw",
    "CAR-MFL": "CAR-MFL",
    "Proposed_CoHiM_FL": "Proposed CoHiM-FL",
}

for metric, ylabel in [
    ("BLEU1", "BLEU-1"),
    ("METEOR", "METEOR"),
    ("Total_Comm_Cost_MB", "Communication Cost (MB)"),
]:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for method in METHODS:
        d = main_summary_numeric[main_summary_numeric["Method"] == method].sort_values("Clients")
        ax.errorbar(
            d["Clients"], d[f"{metric}_Mean"], yerr=d[f"{metric}_SD"],
            marker="o", linewidth=2, capsize=4,
            label=DISPLAY_NAMES.get(method, method),
        )
    ax.set_xlabel("Number of Federated Clients / Medical Institutions")
    ax.set_ylabel(ylabel)
    ax.set_xticks(CLIENT_SETTINGS)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    out = os.path.join(FIGURE_DIR, f"mean_sd_{metric}.png")
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

# Supervisor-requested communication-quality trade-off: FedAvg vs Proposed.
for metric, ylabel in [("BLEU1", "BLEU-1"), ("METEOR", "METEOR")]:
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for method in ["FedAvg", "Proposed_CoHiM_FL"]:
        d = main_summary_numeric[main_summary_numeric["Method"] == method].sort_values("Total_Comm_Cost_MB_Mean")
        ax.errorbar(
            d["Total_Comm_Cost_MB_Mean"], d[f"{metric}_Mean"],
            xerr=d["Total_Comm_Cost_MB_SD"], yerr=d[f"{metric}_SD"],
            marker="o", linewidth=2, capsize=4,
            label=DISPLAY_NAMES.get(method, method),
        )
        for _, row in d.iterrows():
            ax.annotate(str(int(row["Clients"])),
                        (row["Total_Comm_Cost_MB_Mean"], row[f"{metric}_Mean"]),
                        xytext=(5,5), textcoords="offset points", fontsize=8)
    ax.set_xlabel("Communication Cost (MB), Mean ± SD")
    ax.set_ylabel(f"{ylabel}, Mean ± SD")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    out = os.path.join(FIGURE_DIR, f"tradeoff_comm_vs_{metric}.png")
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)


In [ ]:
# =========================================================
# 6) Optional: aggregate missing-modality tables across seeds if all exist
# =========================================================
missing_paths = [
    os.path.join(SPLIT_RUN_ROOT, f"seed_{s}", "missing_modality_results", "01_raw_missing_modality_all_protocols.csv")
    for s in REQUIRED_SEEDS
]

missing_modality_summary_numeric = None
if all(os.path.isfile(p) for p in missing_paths):
    frames = []
    for seed, path in zip(REQUIRED_SEEDS, missing_paths):
        d = canonicalize_legacy_labels(pd.read_csv(path))
        d["Training_Seed"] = seed
        frames.append(d)
    missing_all_seeds = pd.concat(frames, ignore_index=True)
    miss_metrics = [m for m in [
        "BLEU1", "BLEU4", "ROUGE_L", "METEOR", "Label_F1_micro",
        "Label_F1_weighted", "Jaccard_micro", "Hamming_Loss"
    ] if m in missing_all_seeds.columns]
    group_cols = [c for c in ["Method", "Clients", "Missing_Rate", "Missing_Protocol"] if c in missing_all_seeds.columns]
    m_agg = missing_all_seeds.groupby(group_cols)[miss_metrics].agg(["mean", "std"])
    m_agg.columns = [f"{metric}_{'Mean' if stat=='mean' else 'SD'}" for metric, stat in m_agg.columns]
    missing_modality_summary_numeric = m_agg.reset_index()
    print("Missing-modality five-seed aggregation: COMPLETED")
    display(missing_modality_summary_numeric.head(20))
else:
    print("Missing-modality aggregation skipped because not all five optional files are present.")


In [ ]:
# =========================================================
# 6A) Generated-report CheXbert clinical correctness across five seeds
# =========================================================
CLINICAL_METRICS = [
    "Generated_Report_CheXbert_Precision_micro",
    "Generated_Report_CheXbert_Recall_micro",
    "Generated_Report_CheXbert_F1_micro",
    "Generated_Report_CheXbert_F1_macro",
    "Generated_Report_CheXbert_F1_weighted",
    "Generated_Report_CheXbert_Jaccard_micro",
    "Generated_Report_CheXbert_Hamming_Loss",
    "Generated_Report_CheXbert_Exact_Match",
]
clinical_frames = []
for seed in REQUIRED_SEEDS:
    path = os.path.join(
        SPLIT_RUN_ROOT, f"seed_{seed}", "generated_report_clinical",
        f"generated_report_chexbert_seed_{seed}.csv"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing generated-report clinical result for seed {seed}: {path}"
        )
    d = pd.read_csv(path)
    d = canonicalize_legacy_labels(d)
    if set(d["Seed"].astype(int).unique()) != {seed}:
        raise RuntimeError(f"Clinical result seed provenance mismatch: {path}")
    expected_pairs = {(m,c) for c in CLIENT_SETTINGS for m in METHODS}
    actual_pairs = set(zip(d["Method"].astype(str), d["Clients"].astype(int)))
    if actual_pairs != expected_pairs:
        raise RuntimeError(f"Incomplete clinical result for seed {seed}")
    clinical_frames.append(d)

generated_clinical_all_seeds = pd.concat(clinical_frames, ignore_index=True)
counts = generated_clinical_all_seeds.groupby(["Method","Clients"])["Seed"].nunique()
if not (counts == len(REQUIRED_SEEDS)).all():
    raise RuntimeError("Clinical Mean ± SD requires all five seeds in every group.")

clinical_agg = generated_clinical_all_seeds.groupby(["Method","Clients"])[CLINICAL_METRICS].agg(["mean","std"])
clinical_agg.columns = [f"{m}_{'Mean' if s=='mean' else 'SD'}" for m,s in clinical_agg.columns]
generated_clinical_summary_numeric = clinical_agg.reset_index()
generated_clinical_summary_numeric.insert(2, "N_Seeds", len(REQUIRED_SEEDS))

clinical_publication_table = generated_clinical_summary_numeric[["Method","Clients","N_Seeds"]].copy()
for metric in CLINICAL_METRICS:
    clinical_publication_table[metric] = [
        mean_sd_string(m,sd) for m,sd in zip(
            generated_clinical_summary_numeric[f"{metric}_Mean"],
            generated_clinical_summary_numeric[f"{metric}_SD"],
        )
    ]

print("GENERATED-REPORT CHEXBERT FIVE-SEED AGGREGATION: PASSED")
display(clinical_publication_table.sort_values(["Clients","Method"]).reset_index(drop=True))


In [ ]:
# =========================================================
# 6B) Missing-reconstruction stress ablation across five seeds
# =========================================================
RECON_VARIANTS = ["Full_Proposed", "w_o_Missing_Reconstruction"]
RECON_RATES = [0.0, 0.30, 0.60, 0.90]
RECON_METRICS = [
    "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
    "Label_F1_micro", "Label_F1_weighted", "Jaccard_micro", "Hamming_Loss",
]
recon_frames = []
for seed in REQUIRED_SEEDS:
    path = os.path.join(
        SPLIT_RUN_ROOT, f"seed_{seed}", "reconstruction_missing_ablation",
        f"reconstruction_missing_ablation_mean_seed_{seed}.csv"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing reconstruction stress result for seed {seed}: {path}"
        )
    d = pd.read_csv(path)
    if set(d["Seed"].astype(int).unique()) != {seed}:
        raise RuntimeError(f"Reconstruction stress seed mismatch: {path}")
    expected = {(v,float(r)) for v in RECON_VARIANTS for r in RECON_RATES}
    actual = set(zip(d["Ablation_Variant"].astype(str), d["Missing_Rate"].astype(float)))
    if actual != expected:
        raise RuntimeError(
            f"Incomplete reconstruction stress result for seed {seed}. "
            f"Missing={sorted(expected-actual)} Extra={sorted(actual-expected)}"
        )
    recon_frames.append(d)

reconstruction_missing_all_seeds = pd.concat(recon_frames, ignore_index=True)
counts = reconstruction_missing_all_seeds.groupby(["Ablation_Variant","Missing_Rate"])["Seed"].nunique()
if not (counts == len(REQUIRED_SEEDS)).all():
    raise RuntimeError("Reconstruction stress Mean ± SD requires all five seeds.")

ragg = reconstruction_missing_all_seeds.groupby(
    ["Clients","Ablation_Variant","Missing_Rate"]
)[RECON_METRICS].agg(["mean","std"])
ragg.columns = [f"{m}_{'Mean' if s=='mean' else 'SD'}" for m,s in ragg.columns]
reconstruction_missing_summary_numeric = ragg.reset_index()
reconstruction_missing_summary_numeric.insert(3, "N_Seeds", len(REQUIRED_SEEDS))

reconstruction_missing_publication_table = reconstruction_missing_summary_numeric[
    ["Clients","Ablation_Variant","Missing_Rate","N_Seeds"]
].copy()
for metric in RECON_METRICS:
    reconstruction_missing_publication_table[metric] = [
        mean_sd_string(m,sd) for m,sd in zip(
            reconstruction_missing_summary_numeric[f"{metric}_Mean"],
            reconstruction_missing_summary_numeric[f"{metric}_SD"],
        )
    ]

# Paired reconstruction benefit across seeds (Full - NoRecon for higher-is-better).
full = reconstruction_missing_all_seeds[
    reconstruction_missing_all_seeds["Ablation_Variant"] == "Full_Proposed"
].copy()
no = reconstruction_missing_all_seeds[
    reconstruction_missing_all_seeds["Ablation_Variant"] == "w_o_Missing_Reconstruction"
].copy()
paired = full.merge(no, on=["Seed","Clients","Missing_Rate"], suffixes=("_Full","_NoRecon"), validate="one_to_one")
benefit_metrics = ["BLEU1","BLEU4","ROUGE_L","METEOR","Label_F1_micro","Label_F1_weighted","Jaccard_micro"]
for metric in benefit_metrics:
    paired[f"Benefit_{metric}"] = paired[f"{metric}_Full"] - paired[f"{metric}_NoRecon"]
paired["Benefit_Hamming_Loss"] = paired["Hamming_Loss_NoRecon"] - paired["Hamming_Loss_Full"]
benefit_cols = [f"Benefit_{m}" for m in benefit_metrics] + ["Benefit_Hamming_Loss"]
bagg = paired.groupby(["Clients","Missing_Rate"])[benefit_cols].agg(["mean","std"])
bagg.columns = [f"{m}_{'Mean' if s=='mean' else 'SD'}" for m,s in bagg.columns]
reconstruction_benefit_summary = bagg.reset_index()

print("RECONSTRUCTION STRESS FIVE-SEED AGGREGATION: PASSED")
display(reconstruction_missing_publication_table)
print("Paired reconstruction benefit (positive = reconstruction helps):")
display(reconstruction_benefit_summary)


In [ ]:
# =========================================================
# 7) Export all combined tables to CSV / Excel
# =========================================================
os.makedirs(COMBINED_DIR, exist_ok=True)

paths = {
    "split_audit": os.path.join(COMBINED_DIR, "00_data_integrity_audit_5seeds.csv"),
    "main_raw": os.path.join(COMBINED_DIR, "01_main_raw_all_5_seeds.csv"),
    "main_numeric": os.path.join(COMBINED_DIR, "02_main_mean_sd_numeric.csv"),
    "main_publication": os.path.join(COMBINED_DIR, "03_main_publication_mean_sd.csv"),
    "ablation_raw": os.path.join(COMBINED_DIR, "04_ablation_raw_all_5_seeds.csv"),
    "ablation_numeric": os.path.join(COMBINED_DIR, "05_ablation_mean_sd_numeric.csv"),
    "ablation_publication": os.path.join(COMBINED_DIR, "06_ablation_publication_mean_sd.csv"),
    "clinical_raw": os.path.join(COMBINED_DIR, "07_generated_report_chexbert_raw_5seeds.csv"),
    "clinical_numeric": os.path.join(COMBINED_DIR, "08_generated_report_chexbert_mean_sd_numeric.csv"),
    "clinical_publication": os.path.join(COMBINED_DIR, "09_generated_report_chexbert_publication.csv"),
    "recon_raw": os.path.join(COMBINED_DIR, "10_reconstruction_missing_raw_5seeds.csv"),
    "recon_numeric": os.path.join(COMBINED_DIR, "11_reconstruction_missing_mean_sd_numeric.csv"),
    "recon_publication": os.path.join(COMBINED_DIR, "12_reconstruction_missing_publication.csv"),
    "recon_benefit": os.path.join(COMBINED_DIR, "13_reconstruction_benefit_mean_sd.csv"),
    "workbook": os.path.join(COMBINED_DIR, "CoHiM-FL_5Seeds_MeanSD_Combined.xlsx"),
}

split_audit_manifest.to_csv(paths["split_audit"], index=False)
final_test_all_seeds.to_csv(paths["main_raw"], index=False)
main_summary_numeric.to_csv(paths["main_numeric"], index=False)
main_publication_table.to_csv(paths["main_publication"], index=False)
ablation_all_seeds.to_csv(paths["ablation_raw"], index=False)
ablation_summary_numeric.to_csv(paths["ablation_numeric"], index=False)
ablation_publication_table.to_csv(paths["ablation_publication"], index=False)
generated_clinical_all_seeds.to_csv(paths["clinical_raw"], index=False)
generated_clinical_summary_numeric.to_csv(paths["clinical_numeric"], index=False)
clinical_publication_table.to_csv(paths["clinical_publication"], index=False)
reconstruction_missing_all_seeds.to_csv(paths["recon_raw"], index=False)
reconstruction_missing_summary_numeric.to_csv(paths["recon_numeric"], index=False)
reconstruction_missing_publication_table.to_csv(paths["recon_publication"], index=False)
reconstruction_benefit_summary.to_csv(paths["recon_benefit"], index=False)

with pd.ExcelWriter(paths["workbook"], engine="openpyxl") as writer:
    split_audit_manifest.to_excel(writer, sheet_name="Data_Integrity_Audit", index=False)
    final_test_all_seeds.to_excel(writer, sheet_name="Main_Raw_5Seeds", index=False)
    main_summary_numeric.to_excel(writer, sheet_name="Main_Mean_SD", index=False)
    main_publication_table.to_excel(writer, sheet_name="Main_Publication", index=False)
    ablation_all_seeds.to_excel(writer, sheet_name="Ablation_Raw_5Seeds", index=False)
    ablation_summary_numeric.to_excel(writer, sheet_name="Ablation_Mean_SD", index=False)
    ablation_publication_table.to_excel(writer, sheet_name="Ablation_Publication", index=False)
    generated_clinical_all_seeds.to_excel(writer, sheet_name="Clinical_Raw_5Seeds", index=False)
    generated_clinical_summary_numeric.to_excel(writer, sheet_name="Clinical_Mean_SD", index=False)
    clinical_publication_table.to_excel(writer, sheet_name="Clinical_Publication", index=False)
    reconstruction_missing_all_seeds.to_excel(writer, sheet_name="ReconMissing_Raw", index=False)
    reconstruction_missing_summary_numeric.to_excel(writer, sheet_name="ReconMissing_Mean_SD", index=False)
    reconstruction_missing_publication_table.to_excel(writer, sheet_name="ReconMissing_Publication", index=False)
    reconstruction_benefit_summary.to_excel(writer, sheet_name="Recon_Benefit", index=False)
    if missing_modality_summary_numeric is not None:
        missing_modality_summary_numeric.to_excel(writer, sheet_name="Missing_Mean_SD", index=False)

print("Combined tables saved in:", COMBINED_DIR)
for k,v in paths.items():
    print(k, "->", v)


In [ ]:
# =========================================================
# 8) Final ZIP archive + integrity check
# =========================================================
archive_path = os.path.join(
    SPLIT_RUN_ROOT,
    f"CoHiM-FL_Combined_5Seeds_MeanSD_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
)

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    # Combined outputs
    for root, dirs, files in os.walk(COMBINED_DIR):
        dirs[:] = [d for d in dirs if d not in {".ipynb_checkpoints", "__pycache__"}]
        for name in files:
            full = os.path.join(root, name)
            rel = os.path.relpath(full, COMBINED_DIR)
            zf.write(full, arcname=os.path.join("combined_5_seeds", rel))

    # Raw seed result tables only (not model checkpoints) for traceability.
    for seed in REQUIRED_SEEDS:
        for relative in [
            os.path.join("data_integrity", f"data_split_leakage_audit_seed_{seed}.json"),
            os.path.join("data_integrity", f"data_split_membership_seed_{seed}.csv"),
            os.path.join("main_results", f"final_test_seed_{seed}.csv"),
            os.path.join("main_results", f"validation_history_seed_{seed}.csv"),
            os.path.join("ablation_results", f"ablation_final_seed_{seed}.csv"),
            os.path.join("ablation_results", f"ablation_history_seed_{seed}.csv"),
            os.path.join("generated_report_clinical", f"generated_report_chexbert_seed_{seed}.csv"),
            os.path.join("reconstruction_missing_ablation", f"reconstruction_missing_ablation_mean_seed_{seed}.csv"),
            os.path.join("reconstruction_missing_ablation", f"reconstruction_missing_ablation_paired_seed_{seed}.csv"),
            os.path.join("model_checkpoints", f"checkpoint_manifest_seed_{seed}.csv"),
        ]:
            full = os.path.join(SPLIT_RUN_ROOT, f"seed_{seed}", relative)
            if os.path.isfile(full):
                zf.write(full, arcname=os.path.join(f"seed_{seed}", relative))

    manifest = "\n".join([
        "CoHiM-FL Five-Seed Combined Results",
        f"Seeds: {REQUIRED_SEEDS}",
        "Statistic: arithmetic mean ± sample standard deviation (ddof=1)",
        "No best-seed selection.",
        "Data-integrity gate: zero UID/image overlap required for every seed.",
        "All five seeds must have an identical fixed split SHA-256 fingerprint.",
        "Exact missing-study counts: round(rate*N), deterministic nested masks.",
        "Generated-report clinical correctness: REAL CheXbert, Mean ± sample SD across five seeds.",
        "Missing-reconstruction stress ablation: 0/30/60/90%, Mean ± sample SD across five seeds.",
        f"Methods: {METHODS}",
        f"Client settings: {CLIENT_SETTINGS}",
        f"Ablation variants: {ABLATION_VARIANTS}",
        f"Created: {datetime.now().isoformat(timespec='seconds')}",
    ])
    zf.writestr("COMBINED_RESULTS_MANIFEST.txt", manifest)

with zipfile.ZipFile(archive_path, "r") as zf:
    bad = zf.testzip()
if bad is not None:
    raise RuntimeError(f"ZIP integrity check failed: {bad}")

print("\n" + "="*90)
print("FIVE-SEED COMBINATION COMPLETED SUCCESSFULLY")
print("="*90)
print("Mean ± SD: sample SD with ddof=1 across exactly five seeds")
print("Final ZIP:", archive_path)
FINAL_COMBINED_ZIP = archive_path
